In [ ]:
# ==============================================================================
# DualBlind AI Benchmark - Colab Node 2 (Reasoning Champion: DeepSeek-R1 14B)
# ==============================================================================
import os, subprocess, time, urllib.request, re

print("1/4 Installing Ollama & system packages...")
subprocess.run("apt-get update -qq && apt-get install -y -qq zstd pciutils curl", shell=True, check=True)
subprocess.run("curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst -o /tmp/ollama.tar.zst", shell=True, check=True)
subprocess.run("tar --zstd -xf /tmp/ollama.tar.zst -C /usr && rm -f /tmp/ollama.tar.zst", shell=True, check=True)

print("2/4 Starting Ollama daemon...")
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_KEEP_ALIVE"] = "24h"
env["OLLAMA_MAX_LOADED_MODELS"] = "1"
env["OLLAMA_NUM_PARALLEL"] = "1"
subprocess.Popen(["ollama", "serve"], env=env)

# Wait for daemon
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/", timeout=1)
        print("✓ Ollama daemon active on port 11434")
        break
    except Exception:
        time.sleep(1)

print("3/4 Pulling deepseek-r1:14b (Reasoning SOTA, 9.0 GB)...")
subprocess.run(["ollama", "pull", "deepseek-r1:14b"], check=True)

print("4/4 Starting Cloudflare Tunnel...")
subprocess.run("curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared", shell=True, check=True)

proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
tunnel_url = None
for line in proc.stdout:
    match = url_pattern.search(line)
    if match:
        tunnel_url = match.group(0)
        print("\n" + "="*65)
        print(f"🎉 COLAB NODE 2 IS LIVE!")
        print(f"Tunnel URL:  {tunnel_url}")
        print(f"Model:       deepseek-r1:14b (14.7B, Q4_K_M)")
        print("="*65 + "\n")
        break

# Keep session running
while True:
    time.sleep(260000)